# Farmable Issue 20 seven-default retrospective backtest

This credential-free notebook runs protocol Amendment 2 from an exact commit. It keeps the published version 1 result archived and writes a separate seven-default result plus tomato price-only forecasts. The runner requires Amendment 2 on `origin/main` before reading real inputs.

In [ ]:
import os
import re
import subprocess
from pathlib import Path

REVISION = os.environ.get("FARMABLE_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    raise ValueError("Set FARMABLE_REVISION to the exact 40-character commit SHA")
REPOSITORY = "https://github.com/ctrl-alt-elite-za/Farmable.git"
WORKSPACE = Path("/content/Farmable")
subprocess.run(["git", "clone", REPOSITORY, str(WORKSPACE)], check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=WORKSPACE, check=True)
subprocess.run(["git", "fetch", "origin", "main"], cwd=WORKSPACE, check=True)

In [ ]:
subprocess.run(["pip", "install", "uv==0.7.21"], check=True)
subprocess.run(["uv", "sync", "--frozen"], cwd=WORKSPACE, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        str(WORKSPACE / ".venv/bin/python"),
        "xlrd==2.0.2",
        "openpyxl==3.1.5",
    ],
    cwd=WORKSPACE,
    check=True,
)

In [ ]:
subprocess.run(
    ["uv", "run", "--no-sync", "pytest", "ml/forecast", "ml/backtest", "-q"],
    cwd=WORKSPACE,
    check=True,
)
subprocess.run(
    ["uv", "run", "--no-sync", "python", "ml/backtest/check_protocol_first.py", "--check-ready"],
    cwd=WORKSPACE,
    check=True,
)

## Registered Amendment 2 real run

Run only after the amendment has merged independently and all 17 audited workbooks are present. Repeat in a fresh runtime and compare artifact bytes before opening the results PR.

In [ ]:
WORKBOOKS = Path("/content/workbooks")
if not WORKBOOKS.is_dir():
    raise FileNotFoundError("Upload the audited public workbooks to /content/workbooks")
completed = subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "ml/backtest/run_seven_default.py",
        "--workbooks",
        str(WORKBOOKS),
        "--output",
        "/content/issue20-seven-default-results",
    ],
    cwd=WORKSPACE,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)
RUN_ID = completed.stdout.strip().splitlines()[-1]
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "ml/backtest/validate_seven_default.py",
        RUN_ID,
        "--output",
        "/content/issue20-seven-default-results",
    ],
    cwd=WORKSPACE,
    check=True,
)